# Baysor segmentation — Stroke MERSCOPE (Fan / CG)

Batch-runs [Baysor](https://github.com/kharchenkolab/Baysor) cell segmentation on the raw MERSCOPE
transcript CSVs in `/Volumes/T7/Stroke_merscop_Fan_CG/transcript files`.

**Input format** (one CSV per region, MERSCOPE `detected_transcripts`-style):

| col | meaning |
|-----|---------|
| `global_x`, `global_y`, `global_z` | micron coordinates (z is the imaging plane index 0–6) |
| `gene` | gene / barcode name (includes `Blank_*` controls) |
| `fov`, `barcode_id`, `transcript_id`, `x`, `y` | metadata (unused) |

**What this notebook does**
1. Discovers every `*.csv` in the input folder (ignores macOS `._` resource-fork files).
2. For each region, runs Baysor with the lab-standard parameters (`scale=4`, `min_molecules_per_cell=50`,
   `n_clusters=3`, `iters=500`), excluding `Blank*` / control barcodes via `config.data.exclude_genes`
   so we never duplicate the (large) raw files.
3. Writes results to `…/baysor_segmentation/<sample>/m50_s4/segmentation*` and **skips** any region
   that already finished — so the loop is resumable if it is interrupted.

**Kernel:** `Baysor-16-Threads 1.10.9` (Julia 1.10, `JULIA_NUM_THREADS=16`).

> ⚠️ This is heavy: 51 regions, some files >1 GB. Expect this to run for many hours. It is safe to stop and
> re-run — finished regions are detected and skipped.

## 1. Environment

In [1]:
using ProgressMeter
ProgressMeter.ijulia_behavior(:append)

import Pkg
Pkg.activate("/Users/christoffer/Baysor")
using Baysor
using DataFrames
using CSV

println("Julia threads: ", Threads.nthreads())   # should be 16 on the Baysor-16-Threads kernel

  Activating project at `~/Baysor`


Julia threads: 16


## 2. Configuration

In [2]:
# --- paths -------------------------------------------------------------------
input_folder = "/Volumes/T7/Stroke_merscop_Fan_CG/transcript files"

# Outputs go to a sibling folder so the raw transcripts stay untouched.
output_root  = "/Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation"
mkpath(output_root)

# --- Baysor parameters (lab standard for these mouse MERSCOPE panels) --------
baysor_params = (
    x_column = :global_x,
    y_column = :global_y,
    z_column = :global_z,        # set force_2d=true below to ignore z
    gene_column = :gene,
    min_molecules_per_cell = 50,
    n_clusters = 3,
    scale = 4.0,                 # expected cell radius, microns
    iters = 500,
)

# Control / blank barcodes to drop (unanchored regex, matched against gene names).
exclude_genes = "^Blank,^NegControl,^NegativeControl,^Unassigned,^None\$,^DeprecatedCodeword,^FalseCode"

# Folder name encoding the params, e.g. m50_s4
param_tag = "m$(baysor_params.min_molecules_per_cell)_s$(Int(baysor_params.scale))"
println("param_tag = ", param_tag)
println("exclude_genes = ", exclude_genes)

param_tag = m50_s4
exclude_genes = ^Blank,^NegControl,^NegativeControl,^Unassigned,^None$,^DeprecatedCodeword,^FalseCode


## 3. Discover input files

Lists every real `.csv` in the input folder, derives a sample name from the file name, and checks whether a
completed segmentation already exists (a non-empty `segmentation.csv` in the output folder).

In [3]:
"""Output prefix for a given sample, e.g. <output_root>/<sample>/m50_s4/segmentation"""
output_prefix(sample::String) = joinpath(output_root, sample, param_tag, "segmentation")

"""A region counts as done if its segmentation.csv exists and is non-empty."""
function segmentation_done(sample::String)::Bool
    seg = output_prefix(sample) * ".csv"
    return isfile(seg) && filesize(seg) > 0
end

# Collect real CSVs (skip macOS AppleDouble `._` files and hidden dotfiles).
all_csvs = sort([f for f in readdir(input_folder)
                 if endswith(lowercase(f), ".csv") && !startswith(f, "._") && !startswith(f, ".")])

to_process = Tuple{String,String}[]   # (sample, full_path)
skipped    = String[]
for f in all_csvs
    sample = splitext(f)[1]
    path   = joinpath(input_folder, f)
    if segmentation_done(sample)
        push!(skipped, sample)
    else
        push!(to_process, (sample, path))
    end
end

println("="^80)
println("Found $(length(all_csvs)) CSV files")
println("  to process : $(length(to_process))")
println("  already done: $(length(skipped))")
println("="^80)
for (s, p) in to_process
    gb = round(filesize(p) / 1e9, digits=2)
    println("  [ ] $s  ($(gb) GB)")
end
for s in skipped
    println("  [x] $s  (done)")
end

Found 51 CSV files
  to process : 51
  already done: 0
  [ ] 1005_glast_dmcao_5d_1  (0.2 GB)
  [ ] 1005_glast_dmcao_5d_2  (0.23 GB)
  [ ] 1149_glast_dmcao_clp_7d_1  (0.82 GB)
  [ ] 1149_glast_dmcao_clp_7d_2  (0.9 GB)
  [ ] 1149_glast_dmcao_clp_7d_3  (0.43 GB)
  [ ] 1149_glast_dmcao_clp_7d_4  (0.18 GB)
  [ ] 1149_glast_dmcao_clp_7d_5  (0.56 GB)
  [ ] 1151_glast_dmcao_lip_7d_1  (0.6 GB)
  [ ] 1151_glast_dmcao_lip_7d_2  (0.24 GB)
  [ ] 122_mrc1_dcmao_1  (0.22 GB)
  [ ] 122_mrc1_dcmao_2  (0.24 GB)
  [ ] 122_mrc1_dcmao_3  (0.09 GB)
  [ ] 139_mrc1_dmcao_5d_1  (1.24 GB)
  [ ] 139_mrc1_dmcao_5d_2  (0.48 GB)
  [ ] 139_mrc1_dmcao_5d_3  (0.35 GB)
  [ ] 141_mrc1_dmcao_14d_1  (0.77 GB)
  [ ] 141_mrc1_dmcao_14d_2  (0.8 GB)
  [ ] 141_mrc1_dmcao_14d_3  (0.31 GB)
  [ ] 141_mrc1_dmcao_14d_4  (0.58 GB)
  [ ] 182_mrc1_uninjured  (0.69 GB)
  [ ] 26_mrc1_tcmao_14d  (0.05 GB)
  [ ] 27_mrc1_uninjured  (0.04 GB)
  [ ] 465_col1a1_tmcao_5d  (0.02 GB)
  [ ] 467_col1a1_uninjured  (0.02 GB)
  [ ] 51_mrc1_dcmao_14d 

## 4. Runner

In [4]:
"""Run Baysor on one transcript CSV, writing to <output_root>/<sample>/<param_tag>/segmentation*"""
function run_baysor(sample::String, input_path::String; params...)
    out_prefix = output_prefix(sample)
    mkpath(dirname(out_prefix))

    println("\n" * "="^80)
    println("Running Baysor on: $sample")
    println("  input : $input_path")
    println("  output: $(out_prefix)*")
    println("="^80)

    # Build a config object so we can set iters + exclude_genes (not plain kwargs of run()).
    cfg = Baysor.Utils.RunOptions()
    cfg.segmentation.iters = Int(params[:iters])
    cfg.data.exclude_genes = exclude_genes
    # cfg.data.force_2d = true   # <- uncomment to segment in 2D (ignore global_z)

    Baysor.CommandLine.run(
        input_path;
        x_column = params[:x_column],
        y_column = params[:y_column],
        z_column = params[:z_column],
        gene_column = params[:gene_column],
        min_molecules_per_cell = params[:min_molecules_per_cell],
        n_clusters = params[:n_clusters],
        scale = params[:scale],
        output = out_prefix,
        config = cfg,
    )

    println("\nCompleted: $sample")
    return dirname(out_prefix)
end

run_baysor

## 5. Run the batch

Processes every pending region. Errors on one region are caught and logged so the rest still run; re-running
the notebook retries only the failed/unfinished ones.

In [ ]:
results = NamedTuple[]

for (i, (sample, path)) in enumerate(to_process)
    println("\n[$(i)/$(length(to_process))] $sample")
    try
        out = run_baysor(sample, path; baysor_params...)
        push!(results, (sample=sample, status="success", output=out))
    catch e
        println("\nERROR on $sample:")
        showerror(stdout, e); println()
        push!(results, (sample=sample, status="error", output=string(e)))
    end
end

n_ok  = count(r -> r.status == "success", results)
n_err = count(r -> r.status == "error", results)
println("\n" * "="^80)
println("BAYSOR BATCH COMPLETE")
println("  succeeded: $n_ok")
println("  errored  : $n_err")
println("  skipped (already done before this run): $(length(skipped))")
println("="^80)
for r in results
    r.status == "error" && println("  ERROR  $(r.sample): $(first(r.output, 200))")
end


[1/51] 1005_glast_dmcao_5d_1

Running Baysor on: 1005_glast_dmcao_5d_1
  input : /Volumes/T7/Stroke_merscop_Fan_CG/transcript files/1005_glast_dmcao_5d_1.csv
  output: /Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation/1005_glast_dmcao_5d_1/m50_s4/segmentation*
[12:46:51] Info: Run Rad3ea4912
[12:46:51] Info: (2026-06-03) Run Baysor v0.7.1
[12:46:51] Info: Using local Baysor build
[12:46:51] Info: Loading data...
[12:46:56] Warning: Genes ^NegControl,^NegativeControl,^Unassigned,^None$,^DeprecatedCodeword,^FalseCode are missing from the data
└ Baysor.DataLoading /Users/christoffer/Baysor/src/data_loading/data.jl:18
[12:46:56] Info: Excluding genes: Blank_0, Blank_1, Blank_10, Blank_11, Blank_12, Blank_13, Blank_14, Blank_15, Blank_16, Blank_17, Blank_18, Blank_19, Blank_2, Blank_20, Blank_21, Blank_22, Blank_23, Blank_24, Blank_25, Blank_26, Blank_27, Blank_28, Blank_29, Blank_3, Blank_30, Blank_31, Blank_32, Blank_33, Blank_34, Blank_35, Blank_36, Blank_37, Blank_38, Blank_39, Bla

Progress:   0%|                                         |  ETA: 4:06:35
                   Iteration: 2
             Max. difference: 0.834
4m   Fraction of probs changed: 0.997


Progress:   0%|                                         |  ETA: 4:13:35
                   Iteration: 3
             Max. difference: 0.87
4m   Fraction of probs changed: 0.997


Progress:   0%|                                         |  ETA: 4:07:12
                   Iteration: 4
             Max. difference: 0.746
4m   Fraction of probs changed: 0.997


Progress:   0%|                                         |  ETA: 4:04:41
                   Iteration: 5
             Max. difference: 0.678
4m   Fraction of probs changed: 0.996


Progress:   0%|                                         |  ETA: 4:01:19
                   Iteration: 6
             Max. difference: 0.645
4m   Fraction of probs changed: 0.996


Progress:   0%|                                         |  ETA: 4:00:36
                   Iteration:

[12:53:43] Info: Algorithm stopped after 328 iterations. Max. probability difference: 0.00223. Converged: true.
[12:53:43] Info: Done
[12:53:44] Info: Initializing algorithm. Scale: 4.0, scale std: 1.0, initial #components: 103838, #molecules: 2595955.
[12:53:49] Info: Using the following additional information about molecules: [:confidence, :cluster]
[12:53:49] Info: Using 3D coordinates


Progress:   0%|▏                                        |  ETA: 0:08:34
         Iteration: 2
    Noise level, %: 0.7
4m   Num. components: 8412


Progress:   1%|▎                                        |  ETA: 0:08:21
         Iteration: 3
    Noise level, %: 19.22
4m   Num. components: 6504


Progress:   1%|▍                                        |  ETA: 0:07:03
         Iteration: 4
    Noise level, %: 3.4
4m   Num. components: 10506


Progress:   1%|▍                                        |  ETA: 0:06:18
         Iteration: 5
    Noise level, %: 1.58
4m   Num. components: 11017


Progress:   1%|▌                                        |  ETA: 0:06:10
         Iteration: 6
    Noise level, %: 13.47
4m   Num. components: 9407


Progress:   1%|▋                                        |  ETA: 0:05:45
         Iteration: 7
    Noise level, %: 2.13
4m   Num. components: 12238


Progress:   2%|▋                                        |  ETA: 0:05:26
         Iteration: 8
    Noise level

[12:57:25] Info: Processing complete.
[12:57:31] Info: Estimating boundary polygons


...

Progress:   1%|▍                                        |  ETA: 0:00:11
         Iteration: 5
    Noise level, %: 27.48
4m   Num. components: 45


Progress:   2%|▉                                        |  ETA: 0:00:10
         Iteration: 11
    Noise level, %: 26.8
4m   Num. components: 46


Progress:   4%|█▌                                       |  ETA: 0:00:09
         Iteration: 18
    Noise level, %: 26.68
4m   Num. components: 37


Progress:   5%|██▏                                      |  ETA: 0:00:08
         Iteration: 26
    Noise level, %: 26.32
4m   Num. components: 37


Progress:   7%|██▊                                      |  ETA: 0:00:08
         Iteration: 34
    Noise level, %: 26.24
4m   Num. components: 33


Progress:   8%|███▍                                     |  ETA: 0:00:08
         Iteration: 41
    Noise level, %: 26.18
4m   Num. components: 34


Progress:   9%|███▉                                     |  ETA: 0:00:08
         Iteration: 47
    Noise level, %:

[09:30:30] Info: Processing complete.
[09:30:30] Info: Estimating boundary polygons
[09:30:30] Info: Saving results to /Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation/751_col1a1_tmcao_5d/m50_s4/segmentation


Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


[09:30:31] Info: All done!

Completed: 751_col1a1_tmcao_5d

[34/51] 790_glast_dmcao_14d_1

Running Baysor on: 790_glast_dmcao_14d_1
  input : /Volumes/T7/Stroke_merscop_Fan_CG/transcript files/790_glast_dmcao_14d_1.csv
  output: /Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation/790_glast_dmcao_14d_1/m50_s4/segmentation*
[09:30:31] Info: Run R43b037ff6
[09:30:31] Info: (2026-06-04) Run Baysor v0.7.1
[09:30:31] Info: Using local Baysor build
[09:30:31] Info: Loading data...
[09:30:42] Warning: Genes ^NegControl,^NegativeControl,^Unassigned,^None$,^DeprecatedCodeword,^FalseCode are missing from the data
└ Baysor.DataLoading /Users/christoffer/Baysor/src/data_loading/data.jl:18
[09:30:43] Info: Excluding genes: Blank_0, Blank_1, Blank_10, Blank_11, Blank_12, Blank_13, Blank_14, Blank_15, Blank_16, Blank_17, Blank_18, Blank_19, Blank_2, Blank_20, Blank_21, Blank_22, Blank_23, Blank_24, Blank_25, Blank_26, Blank_27, Blank_28, Blank_29, Blank_3, Blank_30, Blank_31, Blank_32, Blank_33, Bla

Progress:   0%|                                         |  ETA: 18.19 days
                   Iteration: 2
             Max. difference: 0.776
4m   Fraction of probs changed: 0.957


Progress:   0%|                                         |  ETA: 17.96 days
                   Iteration: 3
             Max. difference: 0.776
4m   Fraction of probs changed: 0.955


Progress:   0%|                                         |  ETA: 17.94 days
                   Iteration: 4
             Max. difference: 0.781
4m   Fraction of probs changed: 0.952


Progress:   0%|                                         |  ETA: 17.84 days
                   Iteration: 5
             Max. difference: 0.724
4m   Fraction of probs changed: 0.949


Progress:   0%|                                         |  ETA: 17.82 days
                   Iteration: 6
             Max. difference: 0.693
4m   Fraction of probs changed: 0.946


Progress:   0%|                                         |  ETA: 17.86 days
          

## 6. Quick sanity check on outputs

In [ ]:
for f in sort(readdir(output_root))
    seg   = joinpath(output_root, f, param_tag, "segmentation.csv")
    stats = joinpath(output_root, f, param_tag, "segmentation_cell_stats.csv")
    if isfile(stats)
        n_cells = countlines(stats) - 1
        println(rpad(f, 40), "  cells = ", n_cells)
    elseif isfile(seg)
        println(rpad(f, 40), "  segmentation.csv present, stats missing")
    end
end